In [1]:
import os
import pandas as pd
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import re
from sklearn.svm import LinearSVC
import spacy
from scipy.sparse import csr_matrix
from scipy.sparse import hstack
from nltk.sentiment import SentimentIntensityAnalyzer
import textstat
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import NMF
from sklearn.decomposition import TruncatedSVD
from sentence_transformers import SentenceTransformer

C:\Users\kolia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Helpers

In [2]:
def clean_function(text):
    # --- 1. Remove photo captions like (Photo by ... /Getty Images) ---
    text = re.sub(r"\(photo by.*?\)", " ", text, flags=re.IGNORECASE)

    # --- 2. Remove common image credit terms ---
    text = re.sub(r"\b(getty images|getty|photo|image|credit|caption)\b", " ", text, flags=re.IGNORECASE)

    # --- 3. Remove known outlet names (EDIT THIS LIST) ---
    outlet_patterns = [
        r"\b(the\s+)?daily\s+beast\b",
        r"\bmother\s+jones\b",
        r"\balternet\b",
        r"\bcounter\s*punch\b",
        r"\bsalon\b",
        r"\b(the\s+)?american\s+conservative\b",
        r"\b(the\s+)?american\s+spectator\b",
        r"\b(the\s+)?daily\s+caller\b",
        r"\b(the\s+)?washington\s+free\s+beacon\b",
        r"\bfree\s+beacon\b",
        r"\bfox\s+news\b",
        r"\bdigital\b",
    ]
    
    for pattern in outlet_patterns:
        text = re.sub(rf"\b{pattern}\b", " ", text, flags=re.IGNORECASE)

    # --- 4. Remove generic publishing artifacts ---
    text = re.sub(r"\b(news digital|subscribe|newsletter|click here)\b", " ", text, flags=re.IGNORECASE)

    # --- 5. Remove extra whitespace ---
    text = re.sub(r"\s+", " ", text).strip()

    return text

### Initialise File Locations

In [3]:
file_paths = {"AlterNet": os.getcwd() + "\\articles\\left\\alter-net\\",
             "CounterPunch": os.getcwd() + "\\articles\\left\\counter-punch\\",
             "DailyBeast": os.getcwd() + "\\articles\\left\\daily-beast\\",
             "MotherJones": os.getcwd() + "\\articles\\left\\mother-jones\\",
             "Salon": os.getcwd() + "\\articles\\left\\salon\\",
             "American_Conservative": os.getcwd() + "\\articles\\right\\american-conservative\\",
             "American_Spectator": os.getcwd() + "\\articles\\right\\american-spectator\\",
             "DailyCaller": os.getcwd() + "\\articles\\right\\daily-caller\\",
             "FoxNews": os.getcwd() + "\\articles\\right\\fox-news\\",
             "The_Free_Beacon": os.getcwd() + "\\articles\\right\\the-free-beacon\\"}

### Import and Format JSON into Pandas DataFrames

In [4]:
path_file_alternet = file_paths["AlterNet"]+"alternet_200_articles.json"
with open(path_file_alternet, 'r', encoding='utf-8') as file:
    alternet = json.load(file)

df_alternet = pd.DataFrame(pd.json_normalize(alternet))
df_alternet = df_alternet.rename(columns={"headline": "title", "body_text": "text", "body_word_count": "word_count", "outlet": "source"})
df_alternet = df_alternet.iloc[:, [0, 5, 6, 3, 1, 4, 7, 8]]
df_alternet = df_alternet.drop(columns=["url", "processed_at"])

In [5]:
path_file_counterpunch = file_paths["CounterPunch"]+"counterpunch_200_articles.json"
with open(path_file_counterpunch, 'r', encoding='utf-8') as file:
    counterpunch = json.load(file)

df_counterpunch = pd.DataFrame(pd.json_normalize(counterpunch))
df_counterpunch = df_counterpunch.rename(columns={"headline": "title", "body": "text", "outlet name": "source"})
df_counterpunch = df_counterpunch.iloc[:, [0, 8, 6, 4, 2, 5, 1, 7, 3]]
df_counterpunch = df_counterpunch.drop(columns=["url", "crawl_timestamp", "publishing_date"])

In [6]:
path_file_DailyBeast = file_paths["DailyBeast"]+"dailybeast_200_articles.json"
with open(path_file_DailyBeast, 'r', encoding='utf-8') as file:
    DailyBeast = json.load(file)

df_DailyBeast = pd.DataFrame(pd.json_normalize(DailyBeast))
df_DailyBeast = df_DailyBeast.rename(columns={"id": "article_id", "headline": "title", "body": "text"})
df_DailyBeast["source"] = "DailyBeast"
df_DailyBeast["label"] = "left"
df_DailyBeast = df_DailyBeast.iloc[:, [0, 7, 8, 4, 2, 5, 1, 3]]
df_DailyBeast = df_DailyBeast.drop(columns=["url", "published_datetime_raw"])

In [7]:
path_file_MotherJones = file_paths["MotherJones"]+"mother_jones_articles.json"
with open(path_file_MotherJones, 'r', encoding='utf-8') as file:
    MotherJones = json.load(file)

df_MotherJones = pd.DataFrame(pd.json_normalize(MotherJones))
df_MotherJones = df_MotherJones.rename(columns={"header": "title", "body_text": "text", "outlet": "source"})
df_MotherJones = df_MotherJones.iloc[:, [0, 4, 5, 2, 1, 3, 6, 7]]
df_MotherJones = df_MotherJones.drop(columns=["url", "published_date"])

In [8]:
path_file_Salon = file_paths["Salon"]+"salon_articles.json"
with open(path_file_Salon, 'r', encoding='utf-8') as file:
    Salon = json.load(file)

df_Salon = pd.DataFrame(pd.json_normalize(Salon))
df_Salon = df_Salon.rename(columns={"header": "title", "body_text": "text", "outlet": "source"})
df_Salon = df_Salon.iloc[:, [0, 4, 8, 2, 1, 3, 5, 6]]
df_Salon = df_Salon.drop(columns=["url", "published_date"])

In [9]:
path_file_American_Conservative = file_paths["American_Conservative"]+"the_american_conservative_200_articles_20260410_231549.json"
with open(path_file_American_Conservative, 'r', encoding='utf-8') as file:
    American_Conservative = json.load(file)

df_American_Conservative = pd.DataFrame(pd.json_normalize(American_Conservative))
df_American_Conservative = df_American_Conservative.rename(columns={"headline": "title", "body_text": "text", "outlet_name": "source"})
df_American_Conservative = df_American_Conservative.iloc[:, [0, 5, 7, 3, 2, 6, 8, 4, 1]]
df_American_Conservative = df_American_Conservative.drop(columns=["url", "publication_date", "crawl_timestamp"])

In [10]:
path_file_American_Spectator = file_paths["American_Spectator"]+"spectator_articles.json"
with open(path_file_American_Spectator, 'r', encoding='utf-8') as file:
    American_Spectator = json.load(file)

df_American_Spectator = pd.DataFrame(pd.json_normalize(American_Spectator))
df_American_Spectator = df_American_Spectator.rename(columns={"headline": "title", "body_text": "text", "outlet_name": "source"})
df_American_Spectator = df_American_Spectator.iloc[:, [0, 5, 7, 2, 1, 6, 3, 8, 4]]
df_American_Spectator = df_American_Spectator.drop(columns=["url", "publication_date", "crawl_time"])

In [11]:
path_file_DailyCaller = file_paths["DailyCaller"]+"dailycaller_200_articles.json"
with open(path_file_DailyCaller, 'r', encoding='utf-8') as file:
    DailyCaller = json.load(file)

df_DailyCaller = pd.DataFrame(pd.json_normalize(DailyCaller))
df_DailyCaller = df_DailyCaller.rename(columns={"id":"article_id", "headline": "title", "body": "text", "outlet": "source"})
df_DailyCaller = df_DailyCaller.iloc[:, [0, 9, 7, 5, 2, 6, 1, 3, 4, 8]]
df_DailyCaller = df_DailyCaller.drop(columns=["url", "published", "published_raw", "crawl_timestamp"])

In [12]:
path_file_FoxNews = file_paths["FoxNews"]+"foxnews_200_articles.json"
with open(path_file_FoxNews, 'r', encoding='utf-8') as file:
    FoxNews = json.load(file)

df_FoxNews = pd.DataFrame(pd.json_normalize(FoxNews))
df_FoxNews = df_FoxNews.rename(columns={"headline": "title", "body_text": "text", "outlet": "source", "body_word_count": "word_count"})
df_FoxNews = df_FoxNews.iloc[:, [0, 4, 5, 2, 1, 3, 6, 7, 8]]
df_FoxNews = df_FoxNews.drop(columns=["url", "publication_datetime", "crawl_timestamp"])

In [13]:
path_file_The_Free_Beacon = file_paths["The_Free_Beacon"]+"freebeacon_articles.json"
with open(path_file_The_Free_Beacon, 'r', encoding='utf-8') as file:
    The_Free_Beacon = json.load(file)

df_The_Free_Beacon = pd.DataFrame(pd.json_normalize(The_Free_Beacon))
df_The_Free_Beacon = df_The_Free_Beacon.rename(columns={"headline": "title", "main_body": "text", "outlet": "source"})
df_The_Free_Beacon = df_The_Free_Beacon.iloc[:, [4, 6, 7, 2, 0, 5, 1, 3, 8]]
df_The_Free_Beacon = df_The_Free_Beacon.drop(columns=["url", "date_published", "crawl_timestamp"])

### Combine DataFrames and Encode Labels.

**Left => 0
Right => 1**

In [14]:
df_list = [df_alternet, df_counterpunch, df_DailyBeast, df_MotherJones, df_Salon,
           df_American_Conservative, df_American_Spectator, df_DailyCaller, df_FoxNews, df_The_Free_Beacon]

df_final = pd.concat(df_list, axis=0)

In [15]:
df_final.loc[df_final['label']=="left", 'label'] = 0
df_final.loc[df_final['label']=="right", 'label'] = 1

In [92]:
df_final['label'].value_counts()

label
1    1000
0     999
Name: count, dtype: int64

### Split into Train and Test Splits and Clean

**8 outlets for train (4 left- and 4 right-wing)**

**2 outlets for test (1 left- and 1 right-wing)**

In [17]:
df_train = df_final[df_final['source'].isin(["AlterNet", "CounterPunch", "DailyBeast", "Mother Jones",
                                            "The American Conservative", "The American Spectator", "Fox News", "The Daily Caller"
                                           ])]

df_test = df_final[df_final['source'].isin(["Salon", "The Washington Free Beacon"])]

In [18]:
X_train = df_train.loc[:, df_train.columns != 'label']
y_train = df_train['label']
X_test = df_test.loc[:, df_test.columns != 'label']
y_test = df_test['label']
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [19]:
print(X_train.shape, y_train.shape)

(1600, 5) (1600,)


In [20]:
print(X_test.shape, y_test.shape)

(399, 5) (399,)


In [21]:
X_train_text = X_train["text"].apply(clean_function)
X_test_text = X_test["text"].apply(clean_function)

### Transform Train and Test Sets into TF-IDF Metrics

In [22]:
vectorizer = TfidfVectorizer(
    ngram_range = (1, 2),
    max_features = 30000,
    min_df = 3,
    max_df = 0.85,
    stop_words = "english"
)

In [23]:
X_train_tf_idf = vectorizer.fit_transform(X_train_text)
X_test_tf_idf = vectorizer.transform(X_test_text)

## TF-IDF -> Logistic Regression - Baseline Predictor

In [25]:
model_init_lr = LogisticRegression(max_iter = 1000)
model_init_lr.fit(X_train_tf_idf, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [26]:
y_pred_init_lr = model_init_lr.predict(X_test_tf_idf)

In [27]:
print("Accuracy: ", accuracy_score(y_test, y_pred_init_lr))
print(classification_report(y_test, y_pred_init_lr))

Accuracy:  0.7794486215538847
              precision    recall  f1-score   support

           0       0.79      0.76      0.78       199
           1       0.77      0.80      0.78       200

    accuracy                           0.78       399
   macro avg       0.78      0.78      0.78       399
weighted avg       0.78      0.78      0.78       399



## Words that have the most influence on Prediction for Logistic Regression

In [28]:
feature_names = vectorizer.get_feature_names_out()
coefficients = model_init_lr.coef_[0]

In [29]:
top_right = sorted(zip(coefficients, feature_names), reverse=True)[:20]
top_left = sorted(zip(coefficients, feature_names))[:20]

In [30]:
print("Right-Leaning Indicators:")
for coef, word in top_right:
    print(word, coef)

print("\nLeft-Leaning Indicators:")
for coef, word in top_left:
    print(word, coef)

Right-Leaning Indicators:
california 1.2045654703476685
american 1.0851651553317614
democrats 0.9887462366220072
newsom 0.913195848073476
europe 0.9114605179140074
virginia 0.9025096414689532
authorities 0.8814834554588394
city 0.878003360702839
britain 0.8692528894943535
party 0.8610895670067927
foreign 0.8574997763677117
game 0.8436302910624178
european 0.817506361969142
ukraine 0.805331508198195
senate 0.7884273473169036
hamas 0.7675435091598618
regime 0.7492087657370489
america 0.712490649290686
operation 0.6873330235324911
foreign policy 0.6861173992739789

Left-Leaning Indicators:
trump -4.782641740816265
war -1.3972178273444789
hegseth -1.1701936318977382
maga -1.1631176744322498
people -1.113733367905209
war iran -0.9639399901035949
white -0.944913471686754
climate -0.9338421778038288
noem -0.9213522544769372
white house -0.9183120935990093
melania -0.915147446245556
pentagon -0.8807027748242081
year -0.8771614247999056
election -0.8494929903711783
epstein -0.8370315962764251
s

## TF-IDF -> Support Vector Classifier (SVC) - More Sophisticated Classification Model

In [31]:
model_svc_tfidf = LinearSVC()
model_svc_tfidf.fit(X_train_tf_idf, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [32]:
y_pred_svc_tfidf = model_svc_tfidf.predict(X_test_tf_idf)

In [37]:
print("Accuracy: ", accuracy_score(y_test, y_pred_svc_tfidf))
print(classification_report(y_test, y_pred_svc_tfidf))

Accuracy:  0.7744360902255639
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       199
           1       0.77      0.79      0.78       200

    accuracy                           0.77       399
   macro avg       0.77      0.77      0.77       399
weighted avg       0.77      0.77      0.77       399



## Words that have the most influence on Prediction for Support Vector Classifier (SVC)

In [34]:
feature_names_svc_tfidf = vectorizer.get_feature_names_out()
coefficients_svc_tfidf = model_svc_tfidf.coef_[0]

In [35]:
top_right_svc = sorted(zip(coefficients_svc_tfidf, feature_names_svc_tfidf), reverse=True)[:20]
top_left_svc = sorted(zip(coefficients_svc_tfidf, feature_names_svc_tfidf))[:20]

In [36]:
print("Right-Leaning Indicators:")
for coef_svc, word_svc in top_right_svc:
    print(word_svc, coef_svc)

print("\nLeft-Leaning Indicators:")
for coef_svc, word_svc in top_left_svc:
    print(word_svc, coef_svc)

Right-Leaning Indicators:
hamas 1.1382341494277657
california 1.102419935079079
authorities 1.065810908427011
city 1.0564882203676453
american 0.988221707769756
trump said 0.9852861842430178
america 0.9395319391957146
contributed 0.9278630120980953
democrats 0.9177345957539669
regime 0.9093889554919071
great 0.9080815490358316
europe 0.901777198347807
foreign 0.8837629202067925
foreign policy 0.839979119602008
stated 0.8369544479147166
japan 0.8228246079803482
clinton 0.8139547438576868
online 0.811217917958717
virginia 0.8097280554508923
pennsylvania 0.8073795904829753

Left-Leaning Indicators:
trump -3.092677700805173
year -1.1740229892147658
people -1.0603930165156654
explained -0.9595001163023791
maga -0.9435355144958099
workers -0.8934682897174893
war iran -0.8914761005109156
says -0.8860187344142495
war -0.8838626891649044
climate -0.8766654745279852
melania -0.8541439309214319
hegseth -0.8459341933905373
revealed -0.8409531782241919
agents -0.8209640617647219
hours -0.8169399509

## SpaCy POS Tagging -> Combined with TF-IDF into Train + Test Sets

In [38]:
nlp = spacy.load("en_core_web_sm")

In [39]:
modals = {"must", "should", "may", "might", "could", "would", "can"}

In [40]:
def extract_pos_features(text):
    doc = nlp(text)

    total_tokens = 0
    noun = verb = adj = adv = pron = modal = 0

    for token in doc:
        if token.is_punct or token.is_space:
            continue

        total_tokens += 1

        if token.pos_ == "NOUN":
            noun += 1
        elif token.pos_ == "VERB":
            verb += 1
        elif token.pos_ == "ADJ":
            adj += 1
        elif token.pos_ == "ADV":
            adv += 1
        elif token.pos_ == "PRON":
            pron += 1

        # modal detection (by lemma/text)
        if token.lemma_.lower() in modals:
            modal += 1

    if total_tokens == 0:
        return [0, 0, 0, 0, 0, 0]

    return [
        noun / total_tokens,
        verb / total_tokens,
        adj / total_tokens,
        adv / total_tokens,
        pron / total_tokens,
        modal / total_tokens
    ]

In [41]:
X_train_pos = [extract_pos_features(text) for text in X_train_text]
X_test_pos  = [extract_pos_features(text) for text in X_test_text]

In [42]:
X_train_pos = np.array(X_train_pos)
X_test_pos  = np.array(X_test_pos)

In [43]:
print(X_train_pos[0])

[0.19475655 0.11235955 0.07303371 0.02247191 0.09550562 0.00561798]


In [44]:
X_train_pos_sparse = csr_matrix(X_train_pos)
X_test_pos_sparse  = csr_matrix(X_test_pos)

In [46]:
X_train_tfidf_pos = hstack([X_train_tf_idf, X_train_pos_sparse])
X_test_tfidf_pos  = hstack([X_test_tf_idf, X_test_pos_sparse])

## TF-IDF + POS Tag Ratios -> Linear Support Vector Classifier (SVC)

In [47]:
model_tfidf_pos = LinearSVC()
model_tfidf_pos.fit(X_train_combined, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [48]:
y_pred_tfidf_pos = model_tfidf_pos.predict(X_test_combined)

In [49]:
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf_pos))
print(classification_report(y_test, y_pred_tfidf_pos))

Accuracy: 0.7694235588972431
              precision    recall  f1-score   support

           0       0.77      0.77      0.77       199
           1       0.77      0.77      0.77       200

    accuracy                           0.77       399
   macro avg       0.77      0.77      0.77       399
weighted avg       0.77      0.77      0.77       399



#### Accuracy slightly decreased compared to only TF-IDF since some of the POS tag ratios have low predictive power

## Sentiment Feature Engineering -> Combined with TF-IDF and POS Ratios

In [50]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\kolia\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [51]:
sia = SentimentIntensityAnalyzer()

In [52]:
def extract_sentiment_features(text):
    scores = sia.polarity_scores(text)
    
    return [
        scores["neg"],
        scores["neu"],
        scores["pos"],
        scores["compound"]
    ]

In [53]:
X_train_sent = [extract_sentiment_features(text) for text in X_train_text]
X_test_sent  = [extract_sentiment_features(text) for text in X_test_text]

In [54]:
X_train_sent = np.array(X_train_sent)
X_test_sent  = np.array(X_test_sent)

In [55]:
X_train_sent_sparse = csr_matrix(X_train_sent)
X_test_sent_sparse  = csr_matrix(X_test_sent)

X_train_combined2 = hstack([X_train_tf_idf, X_train_pos_sparse, X_train_sent_sparse])
X_test_combined2  = hstack([X_test_tf_idf, X_test_pos_sparse, X_test_sent_sparse])

## Support Vector Classifier with TF-IDF + POS Tag Ratios + Sentiment-Related Features

In [56]:
# SVM
model_sens = LinearSVC()
model_sens.fit(X_train_combined2, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [58]:
y_pred_sens_svc = model_sens.predict(X_test_combined2)

In [60]:
print("Accuracy of SVC on data with engineered sentiment features:", accuracy_score(y_test, y_pred_sens_svc))
print(classification_report(y_test, y_pred_sens_svc))

Accuracy of SVC on data with engineered sentiment features: 0.7644110275689223
              precision    recall  f1-score   support

           0       0.76      0.76      0.76       199
           1       0.77      0.77      0.77       200

    accuracy                           0.76       399
   macro avg       0.76      0.76      0.76       399
weighted avg       0.76      0.76      0.76       399



## Logistic Regression with TF-IDF + POS Tag Ratios + Sentiment-Related Features

In [57]:
# Logistic Regression
model_lr_sens = LogisticRegression(max_iter=1000)
model_lr_sens.fit(X_train_combined2, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [59]:
y_pred_sens_lr = model_lr_sens.predict(X_test_combined2)

In [61]:
print("Accuracy of LR on data with engineered sentiment features:", accuracy_score(y_test, y_pred_sens_lr))
print(classification_report(y_test, y_pred_sens_lr))

Accuracy of LR on data with engineered sentiment features: 0.7669172932330827
              precision    recall  f1-score   support

           0       0.77      0.76      0.76       199
           1       0.76      0.78      0.77       200

    accuracy                           0.77       399
   macro avg       0.77      0.77      0.77       399
weighted avg       0.77      0.77      0.77       399



#### Both for SVC and Logistic Regression, the Accuracy decreased since more noise is added to the already-noisy POS Tag Ratios

## POS Tag Ratios + Sentiment + Readability

In [62]:
def extract_readability_features(text):
    return [
        textstat.flesch_reading_ease(text),
        textstat.flesch_kincaid_grade(text),
        textstat.gunning_fog(text),
        textstat.smog_index(text),
        textstat.automated_readability_index(text),
        textstat.words_per_sentence(text),
    ]

In [63]:
X_train_read = [extract_readability_features(text) for text in X_train_text]
X_test_read  = [extract_readability_features(text) for text in X_test_text]

In [64]:
X_train_read = np.array(X_train_read)
X_test_read  = np.array(X_test_read)

In [65]:
X_train_extra = np.hstack([X_train_pos, X_train_sent, X_train_read])
X_test_extra  = np.hstack([X_test_pos, X_test_sent, X_test_read])

scaler = StandardScaler()

X_train_extra_scaled = scaler.fit_transform(X_train_extra)
X_test_extra_scaled  = scaler.transform(X_test_extra)

X_train_read_sparse = csr_matrix(X_train_extra_scaled)
X_test_read_sparse  = csr_matrix(X_test_extra_scaled)

X_train_final = hstack([
    X_train_tf_idf,
    X_train_read_sparse
])

X_test_final = hstack([
    X_test_tf_idf,
    X_test_read_sparse
])

### Support Vector Classifier (SVC) with POS Tag Ratios + Sentiment + Readability

In [66]:
# SVM
model_all_svm = LinearSVC(C = 0.5, dual = False, max_iter = 20000)
model_all_svm.fit(X_train_final, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,False
,tol,0.0001
,C,0.5
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [67]:
y_pred_all_svm = model_all_svm.predict(X_test_final)

In [68]:
print("Accuracy of SVC on data with all engineered features: ", accuracy_score(y_test, y_pred_all_svm))
print(classification_report(y_test, y_pred_all_svm))

Accuracy of SVC on data with all engineered features:  0.7218045112781954
              precision    recall  f1-score   support

           0       0.71      0.75      0.73       199
           1       0.74      0.69      0.71       200

    accuracy                           0.72       399
   macro avg       0.72      0.72      0.72       399
weighted avg       0.72      0.72      0.72       399



### Logistic Regression with POS Tag Ratios + Sentiment + Readability

In [69]:
# Logistic Regression
model_all_lr = LogisticRegression(max_iter=20000)
model_all_lr.fit(X_train_final, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,20000
,multi_class,'deprecated'


In [70]:
y_pred_all_lr = model_all_lr.predict(X_test_final)

In [71]:
print("Accuracy of LR on data with all engineered features: ", accuracy_score(y_test, y_pred_all_lr))
print(classification_report(y_test, y_pred_all_lr))

Accuracy of LR on data with all engineered features:  0.6390977443609023
              precision    recall  f1-score   support

           0       0.62      0.70      0.66       199
           1       0.66      0.57      0.61       200

    accuracy                           0.64       399
   macro avg       0.64      0.64      0.64       399
weighted avg       0.64      0.64      0.64       399



#### Low-dimensional linguistic features (POS, sentiment, readability) do not significantly improve performance over TF-IDF and may degrade it when combined without careful weighting.

#### In fact, the accuracy metric decreased significantly compared to both models with TF-IDF only and TF-IDF + POS Ratios

## Topic Extraction + TF-IDF

In [72]:
nmf = NMF(n_components=20, random_state=42)

X_train_topics = nmf.fit_transform(X_train_tf_idf)
X_test_topics  = nmf.transform(X_test_tf_idf)

In [73]:
feature_names = vectorizer.get_feature_names_out()

for i, topic in enumerate(nmf.components_):
    top_words = [feature_names[j] for j in topic.argsort()[-10:]]
    print(f"Topic {i}: {top_words}")

Topic 0: ['know', 'don', 'think', 'love', 'time', 'world', 'just', 'life', 'people', 'like']
Topic 1: ['nuclear', 'regime', 'war', 'trump', 'strait hormuz', 'ceasefire', 'hormuz', 'iranian', 'strait', 'iran']
Topic 2: ['donald', 'carlson', 'white', 'white house', 'house', 'iran', 'war', 'maga', 'president', 'trump']
Topic 3: ['survivors', 'statement', 'jeffrey', 'trump', 'files', 'melania trump', 'maxwell', 'lady', 'melania', 'epstein']
Topic 4: ['oil gas', 'china', 'fuel', 'fossil', 'climate', 'prices', 'coal', 'gas', 'energy', 'oil']
Topic 5: ['students', 'city', 'york', 'new york', 'authorities', 'reported', 'school', 'according', 'police', 'said']
Topic 6: ['jewish', 'palestinians', 'palestinian', 'netanyahu', 'kent', 'gaza', 'iran', 'israeli', 'war', 'israel']
Topic 7: ['shutdown', 'enforcement', 'security', 'border', 'tsa', 'noem', 'agents', 'immigration', 'ice', 'dhs']
Topic 8: ['xiv', 'leo xiv', 'meeting', 'cardinal', 'church', 'catholic', 'vatican', 'pope leo', 'leo', 'pope']


In [74]:
X_train_topics_sparse = csr_matrix(X_train_topics)
X_test_topics_sparse  = csr_matrix(X_test_topics)

X_train_final = hstack([X_train_tf_idf, X_train_topics_sparse])
X_test_final  = hstack([X_test_tf_idf, X_test_topics_sparse])

## Support Vector Classifier for Topic-Related Features + TF-IDF

In [75]:
model_svm_topic_tfidf = LinearSVC(C=0.5, dual=False, max_iter=20000)
model_svm_topic_tfidf.fit(X_train_final, y_train)
y_pred_svm_topic_tfidf = model_svm_topic_tfidf.predict(X_test_final)

In [76]:
print("Accuracy of SVC on data with TF-IDF (NMF) and Topic Features: ", accuracy_score(y_test, y_pred_svm_topic_tfidf))
print(classification_report(y_test, y_pred_svm_topic_tfidf))

Accuracy of SVC on data with TF-IDF (NMF) and Topic Features:  0.7744360902255639
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       199
           1       0.77      0.79      0.78       200

    accuracy                           0.77       399
   macro avg       0.77      0.77      0.77       399
weighted avg       0.77      0.77      0.77       399



## Logistic Regression for Topic-Related Features + TF-IDF

In [93]:
# Logistic Regression
model_lr_topic_tfidf = LogisticRegression(max_iter=20000)
model_lr_topic_tfidf.fit(X_train_final, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,20000
,multi_class,'deprecated'


In [94]:
y_pred_lr_topic_tfidf = model_lr_topic_tfidf.predict(X_test_final)

In [95]:
print("Accuracy of LR on data with TF-IDF (NMF) and Topic Features: ", accuracy_score(y_test, y_pred_lr_topic_tfidf))
print(classification_report(y_test, y_pred_lr_topic_tfidf))

Accuracy of LR on data with TF-IDF (NMF) and Topic Features:  0.7844611528822055
              precision    recall  f1-score   support

           0       0.80      0.76      0.78       199
           1       0.77      0.81      0.79       200

    accuracy                           0.78       399
   macro avg       0.78      0.78      0.78       399
weighted avg       0.78      0.78      0.78       399



#### For Support Vector Classifier (SVC) model, compared to when trained and tested on TF-IDF-only data, the accuracy did not change, while for Logistic Regression it improved, increasing from 0.7794486215538847 to 0.7844611528822055

## Topic + TF-IDF (Reduced with SVD)

In [77]:
svd = TruncatedSVD(n_components=200)
X_train_svd = svd.fit_transform(X_train_tf_idf)
X_test_svd  = svd.transform(X_test_tf_idf)

In [78]:
X_train_combined = np.hstack([X_train_svd, X_train_topics])
X_test_combined  = np.hstack([X_test_svd, X_test_topics])

## SVC model for Topic-Related Features + TF-IDF (Reduced with Singular Value Decomposition) 

In [79]:
model = LinearSVC(C=0.5, dual=False, max_iter=20000)
model.fit(X_train_combined, y_train)

y_pred = model.predict(X_test_combined)

In [80]:
print("Accuracy of SVC on data with TF-IDF (SVD) and Topic Features: ", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy of SVC on data with TF-IDF (SVD) and Topic Features:  0.7719298245614035
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       199
           1       0.77      0.79      0.78       200

    accuracy                           0.77       399
   macro avg       0.77      0.77      0.77       399
weighted avg       0.77      0.77      0.77       399



## Logistic Regression for Topic-Related Features + TF-IDF (Reduced with SVD)

In [96]:
# Logistic Regression
model_lr_topic_svd_tfidf = LogisticRegression(max_iter=20000)
model_lr_topic_svd_tfidf.fit(X_train_combined, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,20000
,multi_class,'deprecated'


In [97]:
y_pred_lr_topic_svd_tfidf = model_lr_topic_svd_tfidf.predict(X_test_combined)

In [98]:
print("Accuracy of LR on data with TF-IDF (SVD) and Topic Features: ", accuracy_score(y_test, y_pred_lr_topic_svd_tfidf))
print(classification_report(y_test, y_pred_lr_topic_svd_tfidf))

Accuracy of LR on data with TF-IDF (SVD) and Topic Features:  0.7744360902255639
              precision    recall  f1-score   support

           0       0.78      0.76      0.77       199
           1       0.77      0.79      0.78       200

    accuracy                           0.77       399
   macro avg       0.77      0.77      0.77       399
weighted avg       0.77      0.77      0.77       399



#### For SVC, the accuracy metric decreased only marginally compared to NMF reduction of TF-IDF Features

#### For Logistic Regression, it actually decreased, reaching around the same accuracy as only with TF-IDF Features without dimensionality reduction

## Sentence Embeddings (BERT-like)

In [81]:
model_emb = SentenceTransformer('all-MiniLM-L6-v2')

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [82]:
X_train_emb = model_emb.encode(X_train_text.tolist(), show_progress_bar=True)
X_test_emb  = model_emb.encode(X_test_text.tolist(), show_progress_bar=True)

Batches:   0%|                                                                                  | 0/50 [00:00<?, ?it/s]C:\Users\kolia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|█████████████████████████████████████████████████████████████████████████| 13/13 [00:19<00:00,  1.51s/it]


### SVC For BERT-Like Sentence Embedding-based Features

In [84]:
model_svc_bert = LinearSVC()
model_svc_bert.fit(X_train_emb, y_train)

y_pred_bert = model_svc_bert.predict(X_test_emb)

In [85]:
print("Accuracy of SVC on data with Sentence Embeddings: ", accuracy_score(y_test, y_pred_bert))
print(classification_report(y_test, y_pred_bert))

Accuracy of SVC on data with Sentence Embeddings:  0.6140350877192983
              precision    recall  f1-score   support

           0       0.59      0.71      0.65       199
           1       0.64      0.52      0.57       200

    accuracy                           0.61       399
   macro avg       0.62      0.61      0.61       399
weighted avg       0.62      0.61      0.61       399



### Logistic Regression with Sentence Embedding Features

In [100]:
# Logistic Regression
model_lr_bert = LogisticRegression(max_iter=20000)
model_lr_bert.fit(X_train_emb, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,20000
,multi_class,'deprecated'


In [101]:
y_pred_lr_bert = model_lr_bert.predict(X_test_emb)

In [103]:
print("Accuracy of LR on data with BERT-like Sentence Embedding Features: ", accuracy_score(y_test, y_pred_lr_bert))
print(classification_report(y_test, y_pred_lr_bert))

Accuracy of LR on data with BERT-like Sentence Embedding Features:  0.6641604010025063
              precision    recall  f1-score   support

           0       0.65      0.72      0.68       199
           1       0.69      0.60      0.64       200

    accuracy                           0.66       399
   macro avg       0.67      0.66      0.66       399
weighted avg       0.67      0.66      0.66       399



#### For both SVC and Logistic Regression, the accuracy witnessed a significant decrease. Possible reason: political articles are tied to specific words and jargon. Also, the overall size of the dataset (2000 articles total) is not enough for the models to make sense of it.

## Chunked Sentence Embedding

In [86]:
def chunk_text(text, chunk_size=200):
    words = text.split()
    return [
        " ".join(words[i:i+chunk_size])
        for i in range(0, len(words), chunk_size)
    ]

In [87]:
def embed_document(text, model):
    chunks = chunk_text(text)
    
    embeddings = model.encode(chunks)
    
    return np.mean(embeddings, axis=0)

In [88]:
X_train_emb = np.array([embed_document(text, model_emb) for text in X_train_text])
X_test_emb  = np.array([embed_document(text, model_emb) for text in X_test_text])

C:\Users\kolia\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\nn\modules\module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


## SVC For Chunked Embeddings

In [90]:
model_svc_bert_chunk = LinearSVC()
model_svc_bert_chunk.fit(X_train_emb, y_train)

y_pred_bert_chunk = model_svc_bert_chunk.predict(X_test_emb)

In [91]:
print("Accuracy of SVC on data with Sentence Embeddings (chunked): ", accuracy_score(y_test, y_pred_bert_chunk))
print(classification_report(y_test, y_pred_bert_chunk))

Accuracy of SVC on data with Sentence Embeddings (chunked):  0.6491228070175439
              precision    recall  f1-score   support

           0       0.64      0.69      0.66       199
           1       0.66      0.60      0.63       200

    accuracy                           0.65       399
   macro avg       0.65      0.65      0.65       399
weighted avg       0.65      0.65      0.65       399



## Logistic Regression for Chunked Sentence Embedding Features

In [104]:
# Logistic Regression
model_lr_bert_ch = LogisticRegression(max_iter=20000)
model_lr_bert_ch.fit(X_train_emb, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,20000
,multi_class,'deprecated'


In [106]:
y_pred_lr_bert_ch = model_lr_bert_ch.predict(X_test_emb)

In [107]:
print("Accuracy of LR on data with BERT-like Sentence Embedding Features (Chunked): ", accuracy_score(y_test, y_pred_lr_bert_ch))
print(classification_report(y_test, y_pred_lr_bert_ch))

Accuracy of LR on data with BERT-like Sentence Embedding Features (Chunked):  0.6641604010025063
              precision    recall  f1-score   support

           0       0.65      0.72      0.68       199
           1       0.69      0.60      0.64       200

    accuracy                           0.66       399
   macro avg       0.67      0.66      0.66       399
weighted avg       0.67      0.66      0.66       399



### For political leaning classification with limited data, traditional TF-IDF representations outperform pre-trained sentence embeddings, even when chunking is applied.

→ TF-IDF (1–2 grams)

→ Linear SVM (C ≈ 0.5, dual=False)

→ Cleaned text (no source leakage)

## Final Story

**1. Baseline**

TF-IDF + SVM → strong performance (~77%)

**2. Feature engineering**

POS / sentiment / readability → no improvement

**3. Topic modeling**

Interpretable but no predictive gain

**4. Embeddings**

Underperform due to:

- truncation

- dataset size

- lexical nature of task

**5. Conclusion**

Political leaning is best captured via lexical patterns

Classical models remain competitive